# AutoGluon Forecasting Evaluation Metrics

원본 튜토리얼: https://auto.gluon.ai/dev/tutorials/timeseries/forecasting-metrics.html

이 노트북은 AutoGluon 공식 문서의 `Forecasting Time Series - Evaluation Metrics` 페이지를 바탕으로, 시계열 예측에서 메트릭을 어떻게 선택하고 해석해야 하는지 정리합니다.

이 노트북에서 다루는 핵심은 아래와 같습니다.

- `eval_metric`이 `TimeSeriesPredictor` 학습과 모델 선택에 어떤 영향을 주는지
- point forecast metric과 probabilistic forecast metric의 차이
- scale-dependent, scaled, percentage-based metric을 언제 써야 하는지
- `eval_metric_seasonal_period`와 `quantile_levels`가 왜 중요한지
- `TimeSeriesScorer`를 상속해 custom metric을 만드는 방법

시계열 예측에서는 모델 자체만큼이나 메트릭 선택이 중요합니다. 같은 데이터라도 `MASE`로 학습한 모델과 `RMSE`로 학습한 모델은 다른 예측값을 더 선호할 수 있기 때문입니다.


## 1. 라이브러리 불러오기

이 노트북에서는 `TimeSeriesPredictor`로 간단한 예측기를 만들고, `autogluon.timeseries.metrics`에 포함된 메트릭 클래스로 같은 예측 결과를 여러 기준에서 비교해 봅니다.

- `TimeSeriesDataFrame`: AutoGluon이 이해하는 시계열 데이터 형식
- `TimeSeriesPredictor`: 학습, 예측, 평가를 담당하는 핵심 클래스
- `metrics` 모듈: 내장 forecast metric 클래스 모음
- `TimeSeriesScorer`: custom metric을 만들 때 상속하는 베이스 클래스


In [ ]:
import pandas as pd
import sklearn.metrics

from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
from autogluon.timeseries.metrics import (
    MAE,
    MAPE,
    MASE,
    MSE,
    RMSE,
    RMSLE,
    RMSSE,
    SMAPE,
    SQL,
    WAPE,
    WQL,
    TimeSeriesScorer,
)


## 2. `eval_metric`이 하는 일

공식 문서에서 가장 먼저 강조하는 점은, `eval_metric`이 단순히 마지막에 점수만 보여 주는 옵션이 아니라는 것입니다.

`TimeSeriesPredictor(eval_metric=...)`로 지정한 메트릭은 아래 작업에 직접 영향을 줍니다.

- 모델 하이퍼파라미터 튜닝
- 모델 순위 비교
- 최종 앙상블 구성

즉, 메트릭은 "어떤 예측이 좋은 예측인가"를 AutoGluon에 알려주는 기준입니다.

또 하나 중요한 점은 AutoGluon이 **항상 higher-is-better 형식으로 점수를 보고한다**는 것입니다. 그래서 `MASE`, `RMSE`처럼 원래는 낮을수록 좋은 메트릭도 내부적으로는 부호를 바꿔 `-MASE`, `-RMSE`처럼 표시될 수 있습니다.


In [ ]:
mase_predictor = TimeSeriesPredictor(eval_metric="MASE", prediction_length=3)
wql_predictor = TimeSeriesPredictor(
    eval_metric="WQL",
    prediction_length=3,
    quantile_levels=[0.1, 0.5, 0.9],
)

print("MASE predictor eval_metric:", mase_predictor.eval_metric)
print("WQL predictor eval_metric:", wql_predictor.eval_metric)


## 3. 어떤 메트릭을 골라야 할까?

공식 문서는 메트릭 선택을 위해 세 가지 질문을 먼저 던집니다.

1. point forecast가 필요한가, probabilistic forecast가 필요한가?
2. 값이 큰 시계열을 더 중요하게 볼 것인가?
3. point forecast라면 mean을 맞추고 싶은가, median을 맞추고 싶은가?

이 세 질문만 잘 잡아도 후보가 꽤 좁혀집니다.

- 확률 예측이 중요하다면 `WQL` 또는 `SQL`
- 큰 규모 시계열을 더 중요하게 보고 sparse data가 많다면 `MAE`, `RMSE`, `WAPE`, `WQL`
- 모든 시계열을 비슷한 비중으로 보고 싶다면 `MASE`, `RMSSE`, `SQL`
- 평균을 잘 맞추고 싶다면 `MSE`, `RMSE`, `RMSSE`
- 중앙값을 잘 맞추고 싶다면 `MAE`, `MASE`, `WAPE`

문서는 `MAPE`, `SMAPE` 같은 percentage metric은 한계가 잘 알려져 있어서 실무에서는 보통 우선순위를 낮게 두는 편이 좋다고 설명합니다.


In [ ]:
metric_summary = pd.DataFrame(
    [
        {"metric": "WQL", "type": "probabilistic", "scale_behavior": "scale-dependent", "best_for": "quantile forecast"},
        {"metric": "SQL", "type": "probabilistic", "scale_behavior": "scaled", "best_for": "quantile forecast"},
        {"metric": "MAE", "type": "point", "scale_behavior": "scale-dependent", "best_for": "median"},
        {"metric": "MASE", "type": "point", "scale_behavior": "scaled", "best_for": "median"},
        {"metric": "RMSE", "type": "point", "scale_behavior": "scale-dependent", "best_for": "mean"},
        {"metric": "RMSSE", "type": "point", "scale_behavior": "scaled", "best_for": "mean"},
        {"metric": "WAPE", "type": "point", "scale_behavior": "scale-dependent", "best_for": "median"}
    ]
)

metric_summary


## 4. 실험용 데이터 만들기

메트릭 차이를 직관적으로 보기 위해, 공식 문서의 custom metric 예제와 비슷한 작은 더미 시계열 데이터를 만듭니다.

- 두 개의 시계열을 생성하고
- 마지막 `prediction_length` 구간을 테스트 구간으로 분리한 뒤
- 간단한 `Naive` 모델로 예측을 생성합니다.

이렇게 하면 같은 예측 결과를 여러 메트릭으로 평가해 보기가 쉽습니다.


In [ ]:
data = TimeSeriesDataFrame.from_iterable_dataset(
    [
        {"start": pd.Period("2023-01-01", freq="D"), "target": list(range(15))},
        {"start": pd.Period("2023-01-01", freq="D"), "target": list(range(30, 45))}
    ]
)

prediction_length = 3
train_data, test_data = data.train_test_split(prediction_length=prediction_length)

data


In [ ]:
predictor = TimeSeriesPredictor(
    prediction_length=prediction_length,
    eval_metric="MASE",
    verbosity=0,
).fit(train_data, hyperparameters={"Naive": {}})

predictions = predictor.predict(train_data)
predictions.round(2)


## 5. Point forecast metric 비교하기

이제 같은 `predictions`를 여러 point forecast metric으로 평가해 봅니다.

여기서 보면 이해가 쉬운 포인트가 있습니다.

- `MAE`, `MASE`, `WAPE`는 절대 오차 기반이라 median 쪽 성향이 강합니다.
- `MSE`, `RMSE`, `RMSSE`는 제곱 오차 기반이라 큰 오차를 더 강하게 벌점합니다.
- `MAPE`, `SMAPE`는 비율 기반이라 값의 크기를 상대적으로 보지만, 0이나 작은 값에서 불안정해질 수 있습니다.

같은 예측이라도 어떤 메트릭을 쓰느냐에 따라 "좋다"의 기준이 달라진다는 점이 핵심입니다.


In [ ]:
point_metrics = [
    MAE(prediction_length=prediction_length),
    MAPE(prediction_length=prediction_length),
    MASE(prediction_length=prediction_length),
    MSE(prediction_length=prediction_length),
    RMSE(prediction_length=prediction_length),
    RMSLE(prediction_length=prediction_length),
    RMSSE(prediction_length=prediction_length),
    SMAPE(prediction_length=prediction_length),
    WAPE(prediction_length=prediction_length)
]

point_metric_scores = pd.DataFrame(
    {
        metric.name_with_sign: [
            metric(data=test_data, predictions=predictions, target=predictor.target)
        ]
        for metric in point_metrics
    }
)

point_metric_scores.T.rename(columns={0: "score"})


## 6. `MASE`와 `RMSSE`에서 seasonal period가 중요한 이유

`MASE`, `RMSSE`, `SQL`은 모두 **scaled metric**입니다. 즉, 단순히 절대 오차 크기만 보는 것이 아니라 각 시계열의 과거 변동 규모로 오차를 정규화합니다.

여기서 기준이 되는 값이 `seasonal_period`입니다.

- 일별 데이터의 주간 패턴이면 보통 `7`
- 시간별 데이터의 일간 패턴이면 보통 `24`
- 월별 데이터의 연간 계절성이면 보통 `12`

AutoGluon에서는 이 값을 `eval_metric_seasonal_period`로 지정합니다. seasonal period가 잘못 잡히면 scaled metric의 해석도 흔들릴 수 있으므로, 데이터의 실제 주기를 반영하는 것이 좋습니다.


In [ ]:
scaled_metric_examples = {
    "daily_with_weekly_seasonality": TimeSeriesPredictor(
        prediction_length=7,
        eval_metric="MASE",
        eval_metric_seasonal_period=7,
    ),
    "hourly_with_daily_seasonality": TimeSeriesPredictor(
        prediction_length=24,
        eval_metric="RMSSE",
        eval_metric_seasonal_period=24,
    )
}

scaled_metric_examples


## 7. Probabilistic forecast metric 이해하기

point forecast는 미래를 숫자 하나로 예측합니다. 반면 probabilistic forecast는 불확실성까지 함께 예측합니다.

예를 들어 아래처럼 읽을 수 있습니다.

- `0.1 quantile`: 꽤 낮은 경우 이 정도
- `0.5 quantile`: 중앙값 기준 이 정도
- `0.9 quantile`: 꽤 높은 경우 이 정도

공식 문서에서는 probabilistic metric으로 `WQL`과 `SQL`을 소개합니다.

- `WQL`: 큰 규모 시계열이 더 큰 영향을 주는 quantile loss
- `SQL`: 시계열별 규모를 정규화한 quantile loss

또한 문서에서 중요한 연결점도 알려 줍니다.

- `quantile_levels=[0.5]`이면 `WQL`은 `WAPE`와 연결되고
- `quantile_levels=[0.5]`이면 `SQL`은 `MASE`와 연결됩니다.


In [ ]:
probabilistic_predictor = TimeSeriesPredictor(
    prediction_length=prediction_length,
    eval_metric="WQL",
    quantile_levels=[0.1, 0.5, 0.9],
    verbosity=0,
).fit(train_data, hyperparameters={"Naive": {}})

prob_predictions = probabilistic_predictor.predict(train_data)
prob_predictions.round(2)


In [ ]:
probabilistic_metrics = [
    WQL(prediction_length=prediction_length),
    SQL(prediction_length=prediction_length)
]

prob_metric_scores = pd.DataFrame(
    {
        metric.name_with_sign: [
            metric(
                data=test_data,
                predictions=prob_predictions,
                target=probabilistic_predictor.target,
            )
        ]
        for metric in probabilistic_metrics
    }
)

prob_metric_scores.T.rename(columns={0: "score"})


## 8. `quantile_levels`와 메트릭의 관계

문서에서는 probabilistic forecast를 평가할 때 quantile 자체가 곧 예측 대상이라는 점을 강조합니다. 즉, `WQL`이나 `SQL`을 쓴다는 것은 평균값 하나를 맞히는 것이 아니라 여러 quantile 곡선을 잘 맞히는지를 본다는 뜻입니다.

그래서 `quantile_levels`를 바꾸면 모델이 집중하는 예측 목표도 달라집니다.

- 재고 부족이 더 위험하다면 높은 quantile을 더 중요하게 볼 수 있고
- 비용 절감이 더 중요하다면 중앙값이나 낮은 quantile을 더 자주 볼 수 있습니다.


In [ ]:
quantile_metric_example = TimeSeriesPredictor(
    prediction_length=14,
    eval_metric="WQL",
    quantile_levels=[0.1, 0.5, 0.75, 0.9],
)

quantile_metric_example


## 9. 같은 데이터라도 `eval_metric`에 따라 학습 방향이 달라진다

이제 작은 예제로 `MASE`, `RMSE`, `WQL`을 각각 사용해 예측기를 만들 수 있다는 설정 패턴을 한 번에 정리합니다.

실제 대규모 데이터에서는 이 선택이 더 중요해집니다.

- `MASE`: 모든 시계열을 상대적으로 공평하게 보고 싶을 때
- `RMSE`: 큰 실수에 강한 페널티를 주고 싶을 때
- `WQL`: quantile forecast 자체의 품질이 중요할 때


In [ ]:
metric_configurations = {
    "median_oriented": TimeSeriesPredictor(prediction_length=prediction_length, eval_metric="MASE"),
    "mean_oriented": TimeSeriesPredictor(prediction_length=prediction_length, eval_metric="RMSE"),
    "probabilistic": TimeSeriesPredictor(
        prediction_length=prediction_length,
        eval_metric="WQL",
        quantile_levels=[0.1, 0.5, 0.9],
    )
}

metric_configurations


## 10. Custom forecast metric 만들기

공식 문서에서는 내장 메트릭이 맞지 않을 경우 `TimeSeriesScorer`를 상속해 custom metric을 만들 수 있다고 설명합니다.

핵심 규칙은 아래와 같습니다.

- `compute_metric(data_future, predictions, target, **kwargs)`를 구현해야 합니다.
- 내부 값이 낮을수록 좋은 metric이면 `greater_is_better_internal = False`를 설정합니다.
- quantile 예측이 필요한 metric이면 `needs_quantile = True`를 설정합니다.

또한 문서의 중요한 주의사항도 꼭 기억해야 합니다.

- custom metric 클래스는 실제 사용 시 별도 Python 파일에 두고 import하는 것이 안전합니다.
- 특히 hyperparameter tuning을 켜면 pickling 문제로 notebook 안의 로컬 클래스 정의가 깨질 수 있습니다.


In [ ]:
class MeanSquaredError(TimeSeriesScorer):
    greater_is_better_internal = False
    optimum = 0.0

    def compute_metric(self, data_future, predictions, target, **kwargs):
        return sklearn.metrics.mean_squared_error(
            y_true=data_future[target],
            y_pred=predictions["mean"],
        )


class MeanQuantileLoss(TimeSeriesScorer):
    needs_quantile = True
    greater_is_better_internal = False
    optimum = 0.0

    def compute_metric(self, data_future, predictions, target, **kwargs):
        quantile_columns = [col for col in predictions if col != "mean"]
        total_quantile_loss = 0.0
        for q in quantile_columns:
            total_quantile_loss += sklearn.metrics.mean_pinball_loss(
                y_true=data_future[target],
                y_pred=predictions[q],
                alpha=float(q),
            )
        return total_quantile_loss / len(quantile_columns)


## 11. past data가 필요한 custom metric 예시

문서의 `MASE` 예시는 한 가지 중요한 사실을 보여 줍니다. 어떤 메트릭은 forecast horizon의 오차만으로 계산되지 않고, 과거 시계열 정보도 같이 필요합니다.

대표적으로 `MASE`, `RMSSE`, `SQL`은 과거 seasonal error를 저장해 두었다가 이를 분모로 사용합니다. custom metric에서도 이런 패턴이 필요하면 `save_past_metrics()`와 `clear_past_metrics()`를 구현할 수 있습니다.


In [ ]:
class MeanAbsoluteScaledError(TimeSeriesScorer):
    greater_is_better_internal = False
    optimum = 0.0
    optimized_by_median = True
    equivalent_tabular_regression_metric = "mean_absolute_error"

    def save_past_metrics(
        self,
        data_past: TimeSeriesDataFrame,
        target: str = "target",
        seasonal_period: int = 1,
        **kwargs,
    ) -> None:
        seasonal_diffs = data_past[target].groupby(level="item_id").diff(seasonal_period).abs()
        self._abs_seasonal_error_per_item = seasonal_diffs.groupby(level="item_id").mean().fillna(1.0)

    def clear_past_metrics(self):
        self._abs_seasonal_error_per_item = None

    def compute_metric(self, data_future, predictions, target, **kwargs):
        mae_per_item = (data_future[target] - predictions["mean"]).abs().groupby(level="item_id").mean()
        return (mae_per_item / self._abs_seasonal_error_per_item).mean()


## 12. Custom metric으로 실제 점수 계산하기

`TimeSeriesScorer`를 상속한 metric 객체는 일반 함수처럼 호출할 수 있습니다. AutoGluon이 내부적으로 다음 작업을 알아서 처리합니다.

- `test_data`를 past / future 구간으로 분리
- prediction index 검증
- 필요 시 higher-is-better 형식으로 부호 변환

즉, `compute_metric()`에서는 metric 계산 자체에만 집중하면 됩니다.


In [ ]:
mse_metric = MeanSquaredError(prediction_length=prediction_length)
custom_mse_score = mse_metric(
    data=test_data,
    predictions=predictions,
    target=predictor.target,
)

mql_metric = MeanQuantileLoss(prediction_length=prediction_length)
custom_mql_score = mql_metric(
    data=test_data,
    predictions=prob_predictions,
    target=probabilistic_predictor.target,
)

pd.DataFrame(
    {
        "metric": [mse_metric.name_with_sign, mql_metric.name_with_sign],
        "score": [custom_mse_score, custom_mql_score],
    }
)


## 13. `TimeSeriesPredictor`에서 custom metric 사용하기

문서 마지막 예제처럼 custom metric 인스턴스를 `eval_metric`에 직접 넣어 학습할 수 있습니다.

실전에서는 아래 예시를 notebook 안에 바로 정의한 클래스 대신, 별도 Python 파일로 분리한 뒤 import해서 쓰는 쪽을 권장합니다.


In [ ]:
custom_metric_predictor = TimeSeriesPredictor(
    prediction_length=prediction_length,
    eval_metric=MeanQuantileLoss(),
    verbosity=0,
).fit(
    train_data,
    hyperparameters={"Naive": {}, "SeasonalNaive": {}, "Theta": {}},
)

custom_metric_predictor.leaderboard(test_data)


## 정리

이 노트북은 AutoGluon 공식 `Forecasting Time Series - Evaluation Metrics` 문서를 바탕으로, 시계열 메트릭 선택을 실전 관점에서 다시 정리한 버전입니다.

- `eval_metric`은 단순 표시용 점수가 아니라 학습 방향을 정하는 기준입니다.
- point forecast와 probabilistic forecast는 다른 메트릭을 써야 합니다.
- 큰 시계열을 더 중요하게 볼지, 모든 시계열을 비슷하게 볼지에 따라 scale-dependent와 scaled metric 선택이 달라집니다.
- `MASE`, `RMSSE`, `SQL`을 쓸 때는 `eval_metric_seasonal_period`를 데이터 주기에 맞게 설정하는 것이 중요합니다.
- 내장 메트릭이 맞지 않으면 `TimeSeriesScorer`로 custom metric을 만들 수 있습니다.

처음에는 아래처럼 단순하게 시작하면 좋습니다.

1. point forecast면 `MASE` 또는 `RMSE` 중 하나부터 시작하기
2. quantile forecast가 중요하면 `WQL` 사용하기
3. 이후 데이터 특성과 비즈니스 비용 구조에 맞게 metric을 더 세밀하게 바꾸기
